# Dataset de carreras de Fórmula 1: 2022–2025

Esta recopilación reúne las vueltas de las sesiones de carrera en `df_carreras`. Excluye las sprints y no descarga telemetría ni clima. La primera ejecución puede tardar; cada carrera se guarda individualmente para permitir reanudar el proceso. Comprueba el registro de errores antes de usar el dataset.

In [1]:
import time
from pathlib import Path
import fastf1
import pandas as pd
from IPython.display import display

# Las descargas se guardan por carrera para poder retomar el proceso si se interrumpe.
# La caché de FastF1 se mantiene separada de los DataFrames procesados.
carpeta_cache = Path('cache_fastf1_carreras')
carpeta_vueltas = Path('datos_carreras_2022_2025')
carpeta_cache.mkdir(exist_ok=True)
carpeta_vueltas.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(carpeta_cache))

anios = range(2022, 2026)
tablas_carreras = []
registro_cargas = []
inicio = time.monotonic()

for anio in anios:
    try:
        calendario = fastf1.get_event_schedule(anio, include_testing=False)
    except Exception as error:
        registro_cargas.append({'Temporada': anio, 'Ronda': None, 'GranPremio': None,
                                'Estado': 'Error de calendario', 'Detalle': str(error)})
        print(f'No se pudo cargar el calendario de {anio}: {error}')
        continue

    # Solo eventos cuyo calendario contiene una sesión llamada Race; no incluye sprints.
    columnas_sesion = [f'Session{i}' for i in range(1, 6)]
    carreras = calendario.loc[calendario[columnas_sesion].eq('Race').any(axis=1)]
    for _, evento in carreras.iterrows():
        ronda = int(evento['RoundNumber'])
        nombre = str(evento['EventName'])
        archivo = carpeta_vueltas / f'{anio}_{ronda:02d}.pkl'
        try:
            if archivo.exists():
                vueltas = pd.read_pickle(archivo)
                estado = 'Recuperada'
            else:
                print(f'Cargando {anio}, ronda {ronda}: {nombre}...')
                carrera = fastf1.get_session(anio, ronda, 'R')
                # La tabla de vueltas basta para este primer análisis; no descarga telemetría.
                carrera.load(laps=True, telemetry=False, weather=False, messages=True)
                vueltas = pd.DataFrame(carrera.laps).copy()
                if vueltas.empty:
                    raise ValueError('La sesión no contiene vueltas.')
                vueltas.insert(0, 'Temporada', anio)
                vueltas.insert(1, 'Ronda', ronda)
                vueltas.insert(2, 'GranPremio', nombre)
                vueltas.to_pickle(archivo)
                estado = 'Descargada'
            tablas_carreras.append(vueltas)
            registro_cargas.append({'Temporada': anio, 'Ronda': ronda, 'GranPremio': nombre,
                                    'Estado': estado, 'Detalle': f'{len(vueltas)} vueltas'})
        except Exception as error:
            print(f'No se pudo procesar {anio} - {nombre}: {error}')
            registro_cargas.append({'Temporada': anio, 'Ronda': ronda, 'GranPremio': nombre,
                                    'Estado': 'Error de carrera', 'Detalle': str(error)})

resumen_cargas = pd.DataFrame(registro_cargas)
if not tablas_carreras:
    display(resumen_cargas)
    raise RuntimeError('No se pudo cargar ninguna carrera. Revisa el registro anterior.')

df_carreras = pd.concat(tablas_carreras, ignore_index=True, sort=False)
df_carreras.to_pickle(carpeta_vueltas / 'vueltas_2022_2025.pkl')
print(f'Vueltas reunidas: {len(df_carreras):,} | Carreras: {len(tablas_carreras)} | '
      f'Tiempo: {(time.monotonic() - inicio) / 60:.1f} minutos')
print('Resumen por temporada:')
display(df_carreras.groupby('Temporada').agg(
    carreras=('Ronda', 'nunique'), vueltas=('LapNumber', 'size'),
    pilotos=('Driver', 'nunique')))
print('Carreras o calendarios con errores:')
display(resumen_cargas[resumen_cargas['Estado'].str.startswith('Error')])
print('Primeras vueltas:')
display(df_carreras.head())


/home/simonsloan/anaconda3/lib/python3.13/site-packages/fastf1/req.py:251: FutureWarning: `.enable_cache` is deprecated and will be removed in a future version. Use `.configure` instead.
  warnings.warn("`.enable_cache` is deprecated and will be removed in a "
core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.9.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...


Cargando 2022, ronda 13: Hungarian Grand Prix...


req            INFO 	Data has been written to cache!
core           INFO 	Processing timing data...
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '44', '63', '55', '11', '16', '4', '14', '31', '5', '18', '10', '24', '47', '3', '20', '23', '6', '22', '77']
core           INFO 	Loading data for Belgian Grand Prix - Race [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Cargando 2022, ronda 14: Belgian Grand Prix...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
_api        WARNING 	Failed to align laps for driver

Cargando 2022, ronda 15: Dutch Grand Prix...


... (107 outputs left out)

Carreras o calendarios con errores:


,Temporada,Ronda,GranPremio,Estado,Detalle
56,2024,13.0,Hungarian Grand Prix,Error de carrera,any API: 500 calls/h
57,2024,14.0,Belgian Grand Prix,Error de carrera,any API: 500 calls/h
58,2024,15.0,Dutch Grand Prix,Error de carrera,any API: 500 calls/h
59,2024,16.0,Italian Grand Prix,Error de carrera,any API: 500 calls/h
60,2024,17.0,Azerbaijan Grand Prix,Error de carrera,any API: 500 calls/h
61,2024,18.0,Singapore Grand Prix,Error de carrera,any API: 500 calls/h
62,2024,19.0,United States Grand Prix,Error de carrera,any API: 500 calls/h
63,2024,20.0,Mexico City Grand Prix,Error de carrera,any API: 500 calls/h
64,2024,21.0,São Paulo Grand Prix,Error de carrera,any API: 500 calls/h
65,2024,22.0,Las Vegas Grand Prix,Error de carrera,any API: 500 calls/h


Primeras vueltas:


,Temporada,Ronda,GranPremio,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,2022,1,Bahrain Grand Prix,0 days 01:04:15.340000,VER,1,0 days 00:01:40.236000,1.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:02:34.872000,NaT,1,2.0,False,,False,False
1,2022,1,Bahrain Grand Prix,0 days 01:05:53.220000,VER,1,0 days 00:01:37.880000,2.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:04:15.340000,NaT,1,2.0,False,,False,True
2,2022,1,Bahrain Grand Prix,0 days 01:07:31.577000,VER,1,0 days 00:01:38.357000,3.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:05:53.220000,NaT,1,2.0,False,,False,True
3,2022,1,Bahrain Grand Prix,0 days 01:09:10.143000,VER,1,0 days 00:01:38.566000,4.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:07:31.577000,NaT,1,2.0,False,,False,True
4,2022,1,Bahrain Grand Prix,0 days 01:10:49.020000,VER,1,0 days 00:01:38.877000,5.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:09:10.143000,NaT,1,2.0,False,,False,True


### Recuperar el dataset consolidado



In [2]:
import pandas as pd
from pathlib import Path

archivo = Path("datos_carreras_2022_2025/vueltas_2022_2025.pkl")
if not archivo.exists():
    raise FileNotFoundError("Ejecuta primero la celda de recopilación para generar el dataset.")
df_carreras = pd.read_pickle(archivo)
print(f"{len(df_carreras):,} vueltas cargadas")
display(df_carreras.head())

61,497 vueltas cargadas


,Temporada,Ronda,GranPremio,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,2022,1,Bahrain Grand Prix,0 days 01:04:15.340000,VER,1,0 days 00:01:40.236000,1.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:02:34.872000,NaT,1,2.0,False,,False,False
1,2022,1,Bahrain Grand Prix,0 days 01:05:53.220000,VER,1,0 days 00:01:37.880000,2.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:04:15.340000,NaT,1,2.0,False,,False,True
2,2022,1,Bahrain Grand Prix,0 days 01:07:31.577000,VER,1,0 days 00:01:38.357000,3.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:05:53.220000,NaT,1,2.0,False,,False,True
3,2022,1,Bahrain Grand Prix,0 days 01:09:10.143000,VER,1,0 days 00:01:38.566000,4.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:07:31.577000,NaT,1,2.0,False,,False,True
4,2022,1,Bahrain Grand Prix,0 days 01:10:49.020000,VER,1,0 days 00:01:38.877000,5.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:09:10.143000,NaT,1,2.0,False,,False,True


In [3]:
df_carreras.shape

(61497, 34)

In [9]:
import pandas as pd
import numpy as np

nombres = df_carreras.columns.tolist()
filas = [nombres[i:i + 3] for i in range(0, len(nombres), 3)]

display(pd.DataFrame(filas, columns=['Columna 1', 'Columna 2', 'Columna 3']))

,Columna 1,Columna 2,Columna 3
0,Temporada,Ronda,GranPremio
1,Time,Driver,DriverNumber
2,LapTime,LapNumber,Stint
3,PitOutTime,PitInTime,Sector1Time
4,Sector2Time,Sector3Time,Sector1SessionTime
5,Sector2SessionTime,Sector3SessionTime,SpeedI1
6,SpeedI2,SpeedFL,SpeedST
7,IsPersonalBest,Compound,TyreLife
8,FreshTyre,Team,LapStartTime
9,LapStartDate,TrackStatus,Position
